In [2]:
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
print("project root:", PROJECT_ROOT)

project root: C:\Users\Anshul\Agentic_RAG


In [10]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_core.tools import tool

client = chromadb.PersistentClient(
    path = os.path.join(PROJECT_ROOT, 'chroma_db')
)

embed_fn = SentenceTransformerEmbeddingFunction(model_name = 'all-MiniLM-L6-v2')

collection = client.get_collection(
    name = 'ai_safety_papers',
    embedding_function = embed_fn
)

print('connected, chunks in DB:', collection.count())


connected, chunks in DB: 269


## Retrieval function

In [4]:
def retrieve(query: str, top_k: int=5) -> list[dict]:
    results = collection.query(
        query_texts = [query],
        n_results = top_k,
        include = ['documents', 'metadatas', 'distances']
    )

    chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        chunks.append({
            'text': doc,
            'title': meta.get('title', ''),
            'authors': meta.get('authors', ''),
            'year': meta.get('year', ''),
            'page_num': meta.get('page_num', ''),
            'score': round(1-dist, 4)
        })

    return chunks

In [5]:
chunks = retrieve("how does constitutional AI work?")

for c in chunks:
    print(f"[{c['score']}] {c['title']} — page {c['page_num']}")
    print(f"{c['text'][:150]}\n")

[0.5883] Constitutional AI — page 1
Constitutional AI: Harmlessness from AI Feedback Yuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion, Andy Jones, Anna Chen, 

[0.5402] Constitutional AI — page 5
1.2 The Constitutional AI Approach We will be experimenting with an extreme form of scaled supervision, which we refer to as Constitutional AI (CAI). 

[0.4858] Constitutional AI — page 16
where we update the preference model with new AI feedback in order to keep it on the same distribution as the policy produces. We saw that this was va

[0.4551] AI Safety via Debate — page 22
8 Conclusions and future work We have described debate as an approach to aligning AI systems stronger than humans, and discussed a variety of theoreti

[0.4171] Concrete Problems in AI Safety — page 20
security and safety of systems that interact with the physical world. Illustrative of this work is an impressive and successful eﬀort to formally veri



## Formating the chunks

In [8]:
def format_chunks(chunks: list[dict]) -> str:
    if not chunks:
        return 'No relevant chunks found in the knowledge base.'

    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(
            f"[Chunk {i}] | {c['title']} | {c['authors']} | {c['year']} | Page {c['page_num']} | Score: {c['score']}\n"
            f"{c['text']}"
        )

    return "\n\n---\n\n".join(parts)
        

In [9]:
chunks = retrieve("how does constitutional AI work?")
formatted = format_chunks(chunks)
print(formatted)

[Chunk 1] | Constitutional AI | Bai et al. | 2022 | Page 1 | Score: 0.5883
Constitutional AI: Harmlessness from AI Feedback Yuntao Bai∗, Saurav Kadavath, Sandipan Kundu, Amanda Askell, Jackson Kernion, Andy Jones, Anna Chen, Anna Goldie, Azalia Mirhoseini, Cameron McKinnon, Carol Chen, Catherine Olsson, Christopher Olah, Danny Hernandez, Dawn Drain, Deep Ganguli, Dustin Li, Eli Tran-Johnson, Ethan Perez, Jamie Kerr, Jared Mueller, Jeffrey Ladish, Joshua Landau, Kamal Ndousse, Kamile Lukosuite, Liane Lovitt, Michael Sellitto, Nelson Elhage, Nicholas Schiefer, Noemi Mercado, Nova DasSarma, Robert Lasenby, Robin Larson, Sam Ringer, Scott Johnston, Shauna Kravec, Sheer El Showk, Stanislav Fort, Tamera Lanham, Timothy Telleen-Lawton, Tom Conerly, Tom Henighan, Tristan Hume, Samuel R. Bowman, Zac Hatﬁeld-Dodds, Ben Mann, Dario Amodei, Nicholas Joseph, Sam McCandlish, Tom Brown, Jared Kaplan∗ Anthropic Abstract As AI systems become more capable, we would like to enlist their help to supervise

In [12]:
@tool 
def rag_search(query: str) -> str:
    """
    Search the AI safety research knowledge base containing 6 foundational papers.
    Use this tool when the question is about:
      - Core AI safety concepts (reward hacking, scalable oversight, RLHF)
      - Transformer and attention architecture
      - Constitutional AI methodology
      - Anything that would be in a research paper published before 2023
    
    Do NOT use this for current events, recent news, or anything after 2023.
    """
    chunks = retrieve(query)
    return format_chunks(chunks)


In [13]:
result = rag_search.invoke({"query": "what is scalable oversight?"})
print(result)

[Chunk 1] | Concrete Problems in AI Safety | Amodei et al. | 2016 | Page 15 | Score: 0.5031
an ergodic region of the state space such that actions are reversible [104, 159], or as limiting the probability of huge negative reward to some small value [156]. Yet another approaches uses separate safety and performance functions and attempts to obey constraints on the safety function with high probabilty [22]. As with several of the other directions, applying or adapting these methods to recently developed advanced RL systems could be a promising area of research. This idea seems related to H-inﬁnity control [20] and regional veriﬁcation [148]. • Trusted Policy Oversight: If we have a trusted policy and a model of the environment, we can limit exploration to actions the trusted policy believes we can recover from. It’s ﬁne to dive towards the ground, as long as we know we can pull out of the dive in time. • Human Oversight: Another possibility is to check potentially unsafe actions with a h